In [50]:
import pandas as pd
import altair as alt

In [51]:
library = pd.read_csv("./library_usage.csv")

In [52]:
library.describe()

,Patron Type Code,Total Checkouts,Total Renewals,Circulation Active Year,Year Patron Registered,Supervisor District
count,423448.000000,423448.000000,423448.000000,355544.000000,423448.000000,313138.000000
mean,1.036765,161.982097,59.657327,2014.724943,2010.348917,6.288240
std,4.188198,453.703678,225.009917,1.567938,4.357374,3.123634
min,0.000000,0.000000,0.000000,2003.000000,2003.000000,1.000000
25%,0.000000,2.000000,0.000000,2014.000000,2007.000000,4.000000
50%,0.000000,19.000000,2.000000,2015.000000,2012.000000,6.000000
75%,1.000000,113.000000,27.000000,2016.000000,2014.000000,9.000000
max,104.000000,35907.000000,8965.000000,2016.000000,2016.000000,11.000000


In [53]:
library.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 423448 entries, 0 to 423447
Data columns (total 15 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Patron Type Code              423448 non-null  int64  
 1   Patron Type Definition        423448 non-null  object 
 2   Total Checkouts               423448 non-null  int64  
 3   Total Renewals                423448 non-null  int64  
 4   Age Range                     423233 non-null  object 
 5   Home Library Code             423408 non-null  object 
 6   Home Library Definition       423448 non-null  object 
 7   Circulation Active Month      355544 non-null  object 
 8   Circulation Active Year       355544 non-null  float64
 9   Notice Preference Code        423448 non-null  object 
 10  Notice Preference Definition  423448 non-null  object 
 11  Provided Email Address        423448 non-null  bool   
 12  Year Patron Registered        423448 non-nul

In [54]:
library.sample(1)

,Patron Type Code,Patron Type Definition,Total Checkouts,Total Renewals,Age Range,Home Library Code,Home Library Definition,Circulation Active Month,Circulation Active Year,Notice Preference Code,Notice Preference Definition,Provided Email Address,Year Patron Registered,Outside of County,Supervisor District
127196,4,WELCOME,1,0,20 to 24 years,X,Main Library,March,2016.0,z,email,True,2016,False,7.0


In [55]:
users = library["Patron Type Definition"].value_counts().head(10).index

In [56]:
library = library[library["Patron Type Definition"].isin(users)]

In [57]:
places = library["Home Library Definition"].value_counts().head(15).index

In [58]:
library = library[library["Home Library Definition"].isin(places)]

In [59]:
library = library[library["Outside of County"] == True]

In [60]:
library = library[library["Circulation Active Year"] > 2014]

In [61]:
ages = library["Age Range"].value_counts().head(8).index

In [62]:
library = library[library["Age Range"].isin(ages)]

In [63]:
library = library[library["Total Checkouts"] < 240]

### This is because the data was too big

In [64]:
# library.to_csv("./library_users_clean.csv")

In [65]:
library = library.sample(1000)

In [73]:
alt.Chart(library).mark_bar().encode(
    x=alt.X("Patron Type Definition:O", sort="-y", title='Patron Type'),
    y=alt.Y("count():Q", title='Total Patrons'),
    color=alt.Color("Patron Type Definition:O", scale=alt.Scale(
        scheme='category20b')),  # Set a color scheme
    # Add tooltip for interactivity
    tooltip=['Patron Type Definition', 'count():Q']
).properties(
    width=800,
    height=400,
    title=alt.TitleParams(
        text='Library Patrons by Type',
        subtitle='Total count of patrons based on type',
        align='center',
        fontSize=20
    )
).configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_title(
    fontSize=24,
    anchor='middle',
    color='navy'
)

alt.Chart(...)

In [74]:
alt.Chart(library).mark_circle(size=60, opacity=0.5).encode(
    x=alt.X('Total Checkouts:Q', title='Total Checkouts'),
    y=alt.Y('Total Renewals:Q', title='Total Renewals'),
    color='Age Range:N',  # Optionally color by Age Range to add another dimension
    tooltip=['Patron Type Definition',
             'Total Checkouts', 'Total Renewals', 'Age Range']
).properties(
    width=800,
    height=400,
    title='Total Checkouts vs. Total Renewals'
)

alt.Chart(...)

In [68]:
# plt.figure(figsize=(12, 6))
# sns.boxplot(x='Age Range', y='Total Checkouts', data=library, palette='vlag')
# plt.title('Distribution of Total Checkouts by Age Range')
# plt.xlabel('Age Range')
# plt.ylabel('Total Checkouts')
# plt.xticks(rotation=45)
# plt.grid(True, linestyle='--', linewidth=0.5)
# plt.show()


In [69]:
pivot = library.pivot_table(index='Age Range', values='Total Checkouts')
pivot_long = pivot.reset_index().melt(
    id_vars='Age Range', var_name='Total Checkouts', value_name='Count')
pivot_long

,Age Range,Total Checkouts,Count
0,10 to 19 years,Total Checkouts,30.260870
1,20 to 24 years,Total Checkouts,19.393258
2,25 to 34 years,Total Checkouts,24.860000
3,35 to 44 years,Total Checkouts,42.934132
4,45 to 54 years,Total Checkouts,44.212903
5,55 to 59 years,Total Checkouts,50.707692
6,60 to 64 years,Total Checkouts,49.363636
7,65 to 74 years,Total Checkouts,52.215909


In [75]:
alt.Chart(pivot_long).mark_rect().encode(
    x=alt.X('Total Checkouts:O', title='Total Checkouts'),
    y=alt.Y('Age Range:O', title='Age Range'),
    color=alt.Color('Count:Q', scale=alt.Scale(
        scheme='rainbow'), title='Total Checkouts'),
    tooltip=[
        alt.Tooltip('Total Checkouts:O', title='Total Checkouts'),
        alt.Tooltip('Age Range:O', title='Age Range'),
        alt.Tooltip('Count:Q', title='Count')
    ]
).properties(
    width=800,
    height=400,
    title='Total Checkouts by Age Range'
)

alt.Chart(...)